In [1]:
# backup file reading
fname = "tesla_gateway_backup.bin"
raw = open(fname, 'rb').read()
print(f'The size of the backup file: {len(raw)} bytu')

The size of the backup file: 16777216 bytu


## Partitions layout

```
0x000000-0x020000  mtd0  boot+cfg     (128 KB)   - Bootloader (unchanged)
0x020000-0x200000  mtd1  linux        (1.9 MB)   - Linux kernel
0x200000-0x420000  mtd2  rootfs       (2.1 MB)   - Root filesystem
0x420000-0x1000000 mtd3  jffs2-fs     (11.9 MB)  - User partition
```

In [2]:
parts = ['boot+cfg', 'linux', 'rootfs', 'jffs2-fs'] # names
adrss = [0, 0x020000, 0x200000, 0x420000, 0x1000000] # edge addresses
dlt = 0x80 # how much i'd like to see

# create ranges of data to display
ranges = [[(a-dlt) if (a-dlt) > 0 else 0, (a+dlt) if (a+dlt) < adrss[-1] else adrss[-1]] for a in adrss]

# print one data line
def print_data_row(adr, data):
    print(f'0x{adr:07X}:', end='')
    cnt = 0
    asc = ''
    for d in data:
        print(f"{' ' if cnt % 4 == 0 else ''}{d:02X}", end='')
        asc += chr(d) if 32 <= d <= 126 else '.'
        cnt += 1
    print(f'  {asc}')

B2L = 0x20 # defaultni line length

# print data chunk
def print_data(adr, data, llen=B2L):
    for a in range(len(data) // llen):
        ad = a * llen
        dt = data[ad:ad+llen]
        print_data_row(adr + ad, dt)

# let's print
for e, r in enumerate(ranges):
    print_data(r[0], raw[r[0]:r[1]])
    if r[1] != adrss[-1]:
        print(f'...\n{parts[e]}\n...')

0x0000000: 0BF00004 00000000 00000000 00000000 00004021 40886000 00000000 3C01B800  ..................@!@.`.....<...
0x0000020: 00017825 8DEE0000 00000000 000E7025 3C018196 3421E000 00017825 15CF000A  ..x%..........p%<...4!....x%....
0x0000040: 00000000 3C0FB800 35EF0008 8DEE0000 2401FFFF 01C17024 3C010008 01C17025  ....<...5.......$.....p$<.....p%
0x0000060: ADEE0000 00000000 00000000 00000000 0FF000DA 00000000 00000000 3C013FFF  ............................<.?.
...
boot+cfg
...
0x001FF80: FFFFFFFF FFFFFFFF FFFFFFFF FFFFFFFF FFFFFFFF FFFFFFFF FFFFFFFF FFFFFFFF  ................................
0x001FFA0: FFFFFFFF FFFFFFFF FFFFFFFF FFFFFFFF FFFFFFFF FFFFFFFF FFFFFFFF FFFFFFFF  ................................
0x001FFC0: FFFFFFFF FFFFFFFF FFFFFFFF FFFFFFFF FFFFFFFF FFFFFFFF FFFFFFFF FFFFFFFF  ................................
0x001FFE0: FFFFFFFF FFFFFFFF FFFFFFFF FFFFFFFF FFFFFFFF FFFFFFFF FFFFFFFF FFFFFFFF  ................................
0x0020000: 63723663 80C00000 00020000 00120802 

In [3]:
# load partitions content
pfiles = ['boot.bin', 'kernel.img', 'rootfs.bin', 'userdata.bin']
max_size = [(adrss[i+1] - adrss[i]) for i in range(len(adrss) - 1)]
fcont = {}
for i, s in enumerate(pfiles):
    fcont[s] = open(s, 'rb').read()
    dlen = len(fcont[s])
    dmax = max_size[i]
    print(f"{s:12s} .. {dlen:8d} (0x{dlen:06X}) bytes, {dmax:8d} (0x{dmax:06X}) max{'' if dmax >= dlen else ' .. too big'}")

boot.bin     ..    22646 (0x005876) bytes,   131072 (0x020000) max
kernel.img   ..  1028096 (0x0FB000) bytes,  1966080 (0x1E0000) max
rootfs.bin   ..   479140 (0x074FA4) bytes,  2228224 (0x220000) max
userdata.bin .. 16777216 (0x1000000) bytes, 12451840 (0xBE0000) max .. too big


In [4]:
# what is in userdata and why it's so big
fnam = pfiles[3]
print_data(0, fcont[fnam][:0x100])
print(f'...\n{fnam}\n...')
adr = len(fcont[fnam]) - 0x100
print_data(adr, fcont[fnam][adr:])
    

0x0000000: 626F6F74 00000000 00000000 00005866 0BF00004 00000000 00000000 00000000  boot..........Xf................
0x0000020: 00004025 40886000 00000000 3C01B800 00017825 8DEE0000 00000000 000E7025  ..@%@.`.....<.....x%..........p%
0x0000040: 3C018196 3421E000 00017825 15CF000A 00000000 3C0FB800 35EF0008 8DEE0000  <...4!....x%........<...5.......
0x0000060: 2401FFFF 01C17024 3C010008 01C17025 ADEE0000 00000000 00000000 0FF000D7  $.....p$<.....p%................
0x0000080: 00000000 00000000 3C013FFF 3421FF80 00017025 3C01B800 34211040 00017825  ........<.?.4!....p%<...4!.@..x%
0x00000A0: ADEE0000 00000000 00000000 3C017FFF 3421FF80 00017025 3C01B800 34211040  ............<...4!....p%<...4!.@
0x00000C0: 00017825 ADEE0000 00000000 00000000 3C015080 00017025 3C01B800 34211050  ..x%............<.P...p%<...4!.P
0x00000E0: 00017825 ADEE0000 00000000 00000000 340E0B08 3C01B800 34210010 00017825  ..x%............4...<...4!....x%
...
userdata.bin
...
0x0FFFF00: FFFFFFFF FFFFFFFF FFFFFFFF FFFFF

In [5]:
# building fullimg (naive)
new_fnam = 'new_fullimg.bin'

# glue it together
fullimg = bytearray()
for i, f in enumerate(pfiles):
    max_len = max_size[i]
    dlen = len(fcont[f])
    if dlen < max_len:
        fullimg += fcont[f] + bytearray([0xFF] * (max_len - dlen))
    else:
        fullimg += fcont[f][:max_len]

# print size
print(f'The size of {fnam} is {len(fullimg)}')

#save it
with open(fnam, 'wb') as fd:
    fd.write(fullimg)

The size of userdata.bin is 16777216


In [6]:
# let's print it (to compare)
for e, r in enumerate(ranges):
    print_data(r[0], fullimg[r[0]:r[1]])
    if r[1] != adrss[-1]:
        print(f'...\n{parts[e]}\n...')

0x0000000: 626F6F74 00000000 00000000 00005866 0BF00004 00000000 00000000 00000000  boot..........Xf................
0x0000020: 00004025 40886000 00000000 3C01B800 00017825 8DEE0000 00000000 000E7025  ..@%@.`.....<.....x%..........p%
0x0000040: 3C018196 3421E000 00017825 15CF000A 00000000 3C0FB800 35EF0008 8DEE0000  <...4!....x%........<...5.......
0x0000060: 2401FFFF 01C17024 3C010008 01C17025 ADEE0000 00000000 00000000 0FF000D7  $.....p$<.....p%................
...
boot+cfg
...
0x001FF80: FFFFFFFF FFFFFFFF FFFFFFFF FFFFFFFF FFFFFFFF FFFFFFFF FFFFFFFF FFFFFFFF  ................................
0x001FFA0: FFFFFFFF FFFFFFFF FFFFFFFF FFFFFFFF FFFFFFFF FFFFFFFF FFFFFFFF FFFFFFFF  ................................
0x001FFC0: FFFFFFFF FFFFFFFF FFFFFFFF FFFFFFFF FFFFFFFF FFFFFFFF FFFFFFFF FFFFFFFF  ................................
0x001FFE0: FFFFFFFF FFFFFFFF FFFFFFFF FFFFFFFF FFFFFFFF FFFFFFFF FFFFFFFF FFFFFFFF  ................................
0x0020000: 63733663 80C00000 00020000 000FAC6E 